# Looking Inside MaxSim

MaxSim's decisions are fully transparent, every query token's best match is visible. This notebook uses heatmaps to show exactly how MaxSim disambiguates documents, and why it survives the dilution problem from notebook 02.

In [ ]:
%pip install torch transformers matplotlib seaborn numpy

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
from colbert_from_scratch.viz import plot_maxsim_heatmap

## Setup: Load BERT and define token encoding

We load `bert-base-uncased` and write a function that returns raw token embeddings (768-dim) for any text. No projection, no pooling, just the per-token vectors that BERT produces.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()


def encode_tokens(text):
    """Encode text into per-token BERT embeddings."""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0))
    print(f"'{text}' -> {len(tokens)} tokens, embeddings shape: {embeddings.shape}")
    return embeddings.numpy(), tokens


print("Model loaded.")

## Part A: How to read a MaxSim heatmap

The heatmap is a token-by-token cosine similarity matrix. Rows are query tokens, columns are document tokens. Each cell shows how similar that pair of tokens is in BERT's 768-dimensional embedding space.

The **red rectangles** mark the MaxSim selection: for each query token (row), which document token (column) had the highest similarity. The score bar chart on the right shows each query token's contribution to the total MaxSim score.

A simple example first, a query against a document that contains the answer.

In [ ]:
import json
with open("../data/toy_retrieval.json", "r") as f:
    dataset = json.load(f)

documents = {doc["id"]: doc["text"] for doc in dataset["documents"]}
queries = {q["id"]: q for q in dataset["queries"]}

q0 = queries["q0"]
print(f"Query:    \"{q0['text']}\"")
print(f"Document: \"{documents[0]}\"")

Q_emb, q_tokens = encode_tokens(q0["text"])
D0_emb, d0_tokens = encode_tokens(documents[0])

print(f"\nQuery tokens:    {q_tokens}")
print(f"Document tokens: {d0_tokens}")

In [ ]:
fig = plot_maxsim_heatmap(Q_emb, D0_emb, q_tokens, d0_tokens)
fig.savefig("../figures/03_heatmap_mechanics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
Q_norm = Q_emb / np.linalg.norm(Q_emb, axis=1, keepdims=True)
D_norm = D0_emb / np.linalg.norm(D0_emb, axis=1, keepdims=True)
sim = Q_norm @ D_norm.T

max_values = sim.max(axis=1)
max_indices = sim.argmax(axis=1)

print(f"{'Query Token':<15} {'Best Match':<20} {'Similarity':>10}")
print("-" * 48)
for qt, mv, mi in zip(q_tokens, max_values, max_indices):
    print(f"{qt:<15} {d0_tokens[mi]:<20} {mv:>10.4f}")
print("-" * 48)
print(f"{'TOTAL':<37} {max_values.sum():>10.4f}")

Every content word found its exact match: "python" → "python", "programming" → "programming", "created" → "created", "language" → "language". That's the straightforward case, the document contains the answer and the heatmap lights up on the diagonal-ish matches.

The interesting question: what does this look like when the document is wrong?

---

## Part B: Disambiguation, "python" the language vs. "python" the snake

The word "python" appears in both the programming document (doc 0) and the snake document (doc 1). A single-vector bi-encoder compresses each document into one point, "python" pulls both vectors toward the same region of embedding space.

MaxSim keeps every token separate. The question isn't whether "python" matches, it will, in both docs. The question is whether the other query tokens ("programming", "language", "created") find matches too.

In [ ]:
D1_emb, d1_tokens = encode_tokens(documents[1])

print(f"Query:               \"{q0['text']}\"")
print(f"Doc 0 (relevant):    \"{documents[0][:70]}...\"")
print(f"Doc 1 (hard neg):    \"{documents[1][:70]}...\"")
print(f"\nDoc 0 tokens: {d0_tokens}")
print(f"Doc 1 tokens: {d1_tokens}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

import seaborn as sns

for ax, D_emb, d_tokens, title in [
    (axes[0], D0_emb, d0_tokens, "Doc 0 (relevant): Python programming"),
    (axes[1], D1_emb, d1_tokens, "Doc 1 (hard negative): Python snake"),
]:
    Q_n = Q_emb / np.linalg.norm(Q_emb, axis=1, keepdims=True)
    D_n = D_emb / np.linalg.norm(D_emb, axis=1, keepdims=True)
    s = Q_n @ D_n.T

    sns.heatmap(s, ax=ax, xticklabels=d_tokens, yticklabels=q_tokens,
                cmap="coolwarm", center=0, vmin=-1, vmax=1)

    for i, j in enumerate(s.argmax(axis=1)):
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor="red", linewidth=2))

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Document Tokens")
    ax.set_ylabel("Query Tokens")

fig.suptitle(f'Query: "{q0["text"]}"', fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../figures/03_heatmap_disambiguation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from colbert_from_scratch.maxsim import maxsim_np

score_0 = maxsim_np(Q_emb, D0_emb, normalize=True)
score_1 = maxsim_np(Q_emb, D1_emb, normalize=True)

Q_n = Q_emb / np.linalg.norm(Q_emb, axis=1, keepdims=True)
D0_n = D0_emb / np.linalg.norm(D0_emb, axis=1, keepdims=True)
D1_n = D1_emb / np.linalg.norm(D1_emb, axis=1, keepdims=True)

sim0 = Q_n @ D0_n.T
sim1 = Q_n @ D1_n.T

print(f"MaxSim score vs doc 0 (relevant): {score_0:.4f}")
print(f"MaxSim score vs doc 1 (snake):    {score_1:.4f}")
print(f"Gap: {score_0 - score_1:.4f}")
print()
print(f"{'Query Token':<15} {'Doc 0 match':<18} {'Score':>7}   {'Doc 1 match':<18} {'Score':>7}   {'Diff':>7}")
print("-" * 82)
for i, qt in enumerate(q_tokens):
    m0_val = sim0[i].max()
    m0_tok = d0_tokens[sim0[i].argmax()]
    m1_val = sim1[i].max()
    m1_tok = d1_tokens[sim1[i].argmax()]
    diff = m0_val - m1_val
    print(f"{qt:<15} {m0_tok:<18} {m0_val:>7.4f}   {m1_tok:<18} {m1_val:>7.4f}   {diff:>+7.4f}")

The table tells the whole story. "python" matches strongly in both docs, that token can't disambiguate. But "programming", "language", and "created" find exact matches in doc 0 and only weak, irrelevant matches in doc 1. Those three tokens account for almost the entire score gap.

This is the core mechanism: MaxSim doesn't need every token to discriminate. It only needs some query tokens to have strong matches in the right document and weak matches in the wrong one. The more specific your query terms, the bigger the gap.

A bi-encoder can't show you this. It produces one number, the cosine similarity, with no way to ask "which words contributed to this score?" MaxSim's transparency is a feature, not just a visualization trick.

---

## Part C: The dilution heatmap, seeing the needle in the haystack

In notebook 02, we showed that the bi-encoder's score degrades as irrelevant filler is added around the answer. MaxSim's score barely moves. Now we can see why, the heatmap reveals exactly which tokens MaxSim locks onto, regardless of how much noise surrounds them.

In [ ]:
query_text = "Who created the Python programming language?"

answer = ("The python programming language was created by Guido van Rossum in 1991. "
          "It was designed to be easy to read and simple to implement.")

filler = [
    "Tropical fish require specific water temperatures to thrive in aquariums.",
    "The stock market experienced significant volatility during the first quarter.",
    "Regular exercise has been shown to improve cardiovascular health outcomes.",
    "Ancient Roman architecture influenced building design for centuries.",
    "Cloud computing enables organizations to scale infrastructure on demand.",
    "The periodic table organizes chemical elements by atomic number.",
    "Jazz music originated in New Orleans in the early twentieth century.",
    "Photosynthesis converts sunlight into chemical energy in plant cells.",
]

short_doc = answer
padded_doc = " ".join(filler[:4]) + " " + answer + " " + " ".join(filler[4:])

Q_emb, q_tokens = encode_tokens(query_text)
D_short_emb, d_short_tokens = encode_tokens(short_doc)
D_padded_emb, d_padded_tokens = encode_tokens(padded_doc)

print(f"\nQuery: \"{query_text}\"")
print(f"Short doc:  {len(d_short_tokens)} tokens")
print(f"Padded doc: {len(d_padded_tokens)} tokens")
print(f"\nMaxSim (short):  {maxsim_np(Q_emb, D_short_emb):.4f}")
print(f"MaxSim (padded): {maxsim_np(Q_emb, D_padded_emb):.4f}")

In [ ]:
fig = plot_maxsim_heatmap(Q_emb, D_short_emb, q_tokens, d_short_tokens)
fig.suptitle("Query vs Short Document (answer only)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../figures/03_heatmap_dilution_short.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Heatmap: query vs padded doc (answer buried in filler)
fig = plot_maxsim_heatmap(Q_emb, D_padded_emb, q_tokens, d_padded_tokens)
fig.suptitle("Query vs Padded Document (answer buried in 8 filler sentences)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig("../figures/03_heatmap_dilution_padded.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
Q_n = Q_emb / np.linalg.norm(Q_emb, axis=1, keepdims=True)
D_n = D_padded_emb / np.linalg.norm(D_padded_emb, axis=1, keepdims=True)
sim_padded = Q_n @ D_n.T

print(f"Padded document has {len(d_padded_tokens)} tokens. Answer starts around token ~50.")
print(f"\n{'Query Token':<15} {'Best Match':<18} {'Position':>10} {'Similarity':>12}")
print("-" * 58)
for i, qt in enumerate(q_tokens):
    j = sim_padded[i].argmax()
    print(f"{qt:<15} {d_padded_tokens[j]:<18} {j:>10} {sim_padded[i, j]:>12.4f}")

The red rectangles in the padded heatmap cluster in the same region, the answer sentence. The filler tokens (fish, stocks, exercise, architecture...) appear as a sea of cool blue. MaxSim skips right over them.

This is the visual proof of what notebook 02 showed numerically: the $\max$ operator is immune to document length. It doesn't matter how wide the heatmap gets, the red rectangles find the answer tokens regardless, and the filler tokens contribute nothing to the score.

A bi-encoder would average across the entire width of that heatmap. The bright answer stripe would be diluted into the blue background.

---

## What we learned

1. **MaxSim's decisions are transparent.** Every red rectangle is an explanation, you can point to exactly which document token answered which query token. No other retrieval scoring method offers this level of interpretability.

2. **Disambiguation works through token-level voting.** "python" matches in both the programming and snake documents. The gap comes from the other query tokens, "programming", "language", "created", that only find strong matches in the relevant document.

3. **Dilution immunity is visible.** The padded-document heatmap shows filler as inert noise, cool blue or whiteish red cells that MaxSim ignores. The answer tokens are still the brightest matches, still selected by the red rectangles, still producing the same score.

MaxSim has zero trainable parameters. It's just matrix multiplication + max + sum. Everything you see in these heatmaps comes from BERT's embeddings, MaxSim is a lens that makes BERT's knowledge visible and usable for retrieval.